In [1]:
# ============================================================
# Section 1: Imports and Test 2 settings
# ============================================================

from pathlib import Path
from datetime import datetime

import osxphotos

from explorephotoslibrary import *


USE_INVENTORY_CACHE = True

# For this run, rebuild backup because the backup Photos Library moved
# to a different external SSD path.
#
# After this run succeeds once, change this to:
# FORCE_REBUILD_INVENTORY_KEYS = set()
FORCE_REBUILD_INVENTORY_KEYS = set()

# Set to True only when you want to pick the Photos Library paths again.
# If False, Test 2 reuses paths saved in data/local_config/test2_library_paths.json.
FORCE_RESELECT_LIBRARY_PATHS = False


TEST2_LIBRARY_PROMPTS = {
    "backup_20250317": "Select BACKUP Photos Library: backup_20250317",
    "current_default": "Select CURRENT default Photos Library: current_default",
}


TEST2_DEFAULT_INITIAL_DIRS = {
    "backup_20250317": "/Volumes",
    "current_default": str(Path.home() / "Pictures"),
}

In [2]:
# ============================================================
# Section 2: Load or build inventories
# ============================================================

TEST2_LIBRARY_HISTORY_PATH = Path("data/local_config/test2_library_paths.json")


def get_test2_library_path(library_key):
    library_history = load_json_file(TEST2_LIBRARY_HISTORY_PATH, default={}) or {}

    saved_library_path = library_history.get(library_key)

    if (
        saved_library_path
        and Path(saved_library_path).exists()
        and not FORCE_RESELECT_LIBRARY_PATHS
    ):
        library_path = Path(saved_library_path)

        print("=" * 80)
        print(f"Use saved Photos Library path for: {library_key}")
        print("=" * 80)
        print(f"{library_key} library path:", library_path)
        print()

        return library_path

    if saved_library_path:
        initial_dir = Path(saved_library_path).parent
    else:
        initial_dir = Path(TEST2_DEFAULT_INITIAL_DIRS.get(library_key, "/Volumes"))

    prompt = TEST2_LIBRARY_PROMPTS.get(
        library_key,
        f"Select Photos Library for: {library_key}",
    )

    print("=" * 80)
    print(prompt)
    print("=" * 80)

    library_path = Path(
        choose_photos_library_path(
            initial_dir=initial_dir,
            prompt=prompt,
        )
    )

    library_history[library_key] = str(library_path)
    library_history[f"{library_key}_selected_at"] = datetime.now().isoformat()
    save_json_file(TEST2_LIBRARY_HISTORY_PATH, library_history)

    print(f"{library_key} library path:", library_path)
    print()

    return library_path


def load_or_build_inventory(library_key):
    library_path = get_test2_library_path(library_key)

    should_rebuild_inventory = library_key in FORCE_REBUILD_INVENTORY_KEYS

    if USE_INVENTORY_CACHE and not should_rebuild_inventory:
        print("=" * 80)
        print(f"Load inventory cache: {library_key}")
        print("=" * 80)

        try:
            inventory = load_inventory_cache(library_key)
            return inventory
        except FileNotFoundError:
            print(f"Cache not found for {library_key}. Build inventory instead.")
            print()

    if should_rebuild_inventory:
        print("=" * 80)
        print(f"Force rebuild inventory: {library_key}")
        print("=" * 80)
    else:
        print("=" * 80)
        print(f"Build inventory: {library_key}")
        print("=" * 80)

    osx_assets = osxphotos.PhotosDB(str(library_path)).photos()
    print(f"{library_key} osx asset count:", len(osx_assets))

    inventory = build_inventory(osx_assets)

    print()
    print(f"{library_key} inventory summary")
    print("-" * 80)
    print_inventory_summary(inventory)

    save_inventory_cache(inventory, library_key)

    return inventory


inventory_backup = load_or_build_inventory("backup_20250317")

print()

inventory_current = load_or_build_inventory("current_default")

Use saved Photos Library path for: backup_20250317
backup_20250317 library path: /Volumes/PRO-G40-0605/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary

Load inventory cache: backup_20250317
Cache not found for backup_20250317. Build inventory instead.

Build inventory: backup_20250317
backup_20250317 osx asset count: 71573
processed assets: 10000
processed assets: 20000
processed assets: 30000
processed assets: 40000
processed assets: 50000
processed assets: 60000
processed assets: 70000

backup_20250317 inventory summary
--------------------------------------------------------------------------------
inventory assets: 71573
inventory albums: 5170
inventory folders: 35
special assets:
  PATH_NONE: 0
  SYNDICATED: 0
  UNKNOWN_PATH: 0
movies: 6224
hidden: 0
favorites: 701
descriptions: 727
keywords: 23747
saved inventory cache: data/inven

In [3]:
# ============================================================
# Section 3: Photo Library asset unique ID audit
# ============================================================

print("=" * 120)
print("Section 3: Photo Library asset unique ID audit")
print("=" * 120)

print()
backup_unique_ok = audit_photo_library_asset_unique_ids(
    inventory_backup,
    label="BACKUP",
)

print()
current_unique_ok = audit_photo_library_asset_unique_ids(
    inventory_current,
    label="CURRENT",
)

if not backup_unique_ok:
    raise RuntimeError("BACKUP photo_library_asset_unique_id audit failed.")

if not current_unique_ok:
    raise RuntimeError("CURRENT photo_library_asset_unique_id audit failed.")

print()
print("Photo Library asset unique ID audit passed.")

Section 3: Photo Library asset unique ID audit

BACKUP
--------------------------------------------------------------------------------
total asset count: 71573
generated unique ID count: 71573
assets without unique ID: 0
duplicate unique ID group count: 0
duplicate asset count: 0
is Photo Library asset unique ID scheme unique: True

CURRENT
--------------------------------------------------------------------------------
total asset count: 94698
generated unique ID count: 94698
assets without unique ID: 0
duplicate unique ID group count: 0
duplicate asset count: 0
is Photo Library asset unique ID scheme unique: True

Photo Library asset unique ID audit passed.


In [15]:
# ============================================================
# Section 4: Cross-library comparison helpers
# ============================================================

from enum import Enum
from collections import Counter, defaultdict
import explorephotoslibrary as epl


class ChangeType(str, Enum):
    # Asset existence
    ASSET_MISSING_FROM_CURRENT = "ASSET_MISSING_FROM_CURRENT"
    ASSET_PRESENT_IN_CURRENT_AS_SYNDICATED = "ASSET_PRESENT_IN_CURRENT_AS_SYNDICATED"
    ASSET_NEW_IN_CURRENT = "ASSET_NEW_IN_CURRENT"

    # Asset metadata / organization
    ASSET_METADATA_CHANGED = "ASSET_METADATA_CHANGED"
    ASSET_ALBUM_MEMBERSHIP_CHANGED = "ASSET_ALBUM_MEMBERSHIP_CHANGED"
    ASSET_FOLDER_PATHS_CHANGED = "ASSET_FOLDER_PATHS_CHANGED"

    # Album / folder structure
    ALBUM_MISSING_FROM_CURRENT = "ALBUM_MISSING_FROM_CURRENT"
    ALBUM_NEW_IN_CURRENT = "ALBUM_NEW_IN_CURRENT"
    ALBUM_FOLDER_PATHS_CHANGED = "ALBUM_FOLDER_PATHS_CHANGED"
    FOLDER_MISSING_FROM_CURRENT = "FOLDER_MISSING_FROM_CURRENT"
    FOLDER_NEW_IN_CURRENT = "FOLDER_NEW_IN_CURRENT"


CHANGE_TYPE_DESCRIPTIONS = {
    ChangeType.ASSET_MISSING_FROM_CURRENT:
        "Asset exists in backup normal assets, and no matching current normal asset or current syndicated asset was found.",

    ChangeType.ASSET_PRESENT_IN_CURRENT_AS_SYNDICATED:
        "Asset exists in backup normal assets, and exists in current special_assets/SYNDICATED. This is not true current-missing.",

    ChangeType.ASSET_NEW_IN_CURRENT:
        "Asset exists in current normal assets but not in backup normal assets. Usually normal because current library is later.",

    ChangeType.ASSET_METADATA_CHANGED:
        "Same asset identity exists in both libraries, but metadata differs.",

    ChangeType.ASSET_ALBUM_MEMBERSHIP_CHANGED:
        "Same asset identity exists in both libraries, but album membership differs.",

    ChangeType.ASSET_FOLDER_PATHS_CHANGED:
        "Same asset identity exists in both libraries, but folder paths differ.",

    ChangeType.ALBUM_MISSING_FROM_CURRENT:
        "Album title exists in backup but not in current.",

    ChangeType.ALBUM_NEW_IN_CURRENT:
        "Album title exists in current but not in backup.",

    ChangeType.ALBUM_FOLDER_PATHS_CHANGED:
        "Album title exists in both libraries, but the album folder path differs.",

    ChangeType.FOLDER_MISSING_FROM_CURRENT:
        "Folder path exists in backup but not in current.",

    ChangeType.FOLDER_NEW_IN_CURRENT:
        "Folder path exists in current but not in backup.",
}


def change_type_value(change_type):
    if isinstance(change_type, ChangeType):
        return change_type.value

    return str(change_type)


def change_type_description(change_type):
    if not isinstance(change_type, ChangeType):
        try:
            change_type = ChangeType(change_type)
        except ValueError:
            return ""

    return CHANGE_TYPE_DESCRIPTIONS.get(change_type, "")


def add_diff(
    diff_records,
    change_type,
    scope,
    backup_object=None,
    current_object=None,
    backup_value=None,
    current_value=None,
    note=None,
):
    diff_records.append({
        "change_type": change_type_value(change_type),
        "scope": scope,
        "backup_object": backup_object,
        "current_object": current_object,
        "backup_value": backup_value,
        "current_value": current_value,
        "description": change_type_description(change_type),
        "note": note,
    })


def sort_asset_ids(asset_ids):
    return sorted(asset_ids, key=repr)


def asset_display_name(asset):
    if asset is None:
        return None

    return {
        "uuid": asset.get("uuid"),
        "original_filename": asset.get("original_filename"),
        "filename": asset.get("filename"),
        "date": asset.get("date"),
        "photo_library_asset_unique_id": asset.get("photo_library_asset_unique_id"),
    }


def album_display_name(album):
    if album is None:
        return None

    return {
        "uuid": album.get("uuid"),
        "title": album.get("title"),
        "folder_paths": album_folder_paths(album),
    }


def folder_display_name(folder):
    if folder is None:
        return None

    return {
        "uuid": folder.get("uuid"),
        "title": folder.get("title"),
        "path": folder.get("path"),
    }


def normalize_scalar_value(value):
    if value is None:
        return None

    if isinstance(value, list):
        return tuple(value)

    if isinstance(value, tuple):
        return tuple(value)

    if isinstance(value, dict):
        return repr(value)

    return value


def normalize_string_tuple(values):
    if not values:
        return tuple()

    normalized = []

    for value in values:
        if value is None:
            continue

        text = str(value).strip()

        if text:
            normalized.append(text)

    return tuple(sorted(set(normalized)))


def asset_album_titles(asset):
    albums = asset.get("albums") or {}

    return normalize_string_tuple(
        album.get("title")
        for album in albums.values()
        if album.get("title")
    )


def asset_folder_paths(asset):
    folders = asset.get("folders") or {}

    return normalize_string_tuple(
        (folder.get("path") or folder.get("title"))
        for folder in folders.values()
        if (folder.get("path") or folder.get("title"))
    )


def album_folder_paths(album):
    # Robust version:
    # some album objects may not have "folders".
    folders = album.get("folders") or {}

    return sorted(
        folder.get("path") or folder.get("title")
        for folder in folders.values()
        if folder.get("path") or folder.get("title")
    )


def make_asset_index(inventory):
    index = {}

    for asset in inventory.get("assets") or []:
        unique_id = asset.get("photo_library_asset_unique_id")

        if unique_id is None:
            raise RuntimeError(
                "Normal asset has no photo_library_asset_unique_id:\n"
                f"{asset}"
            )

        if unique_id in index:
            raise RuntimeError(
                "Duplicate photo_library_asset_unique_id while making asset index:\n"
                f"{unique_id}"
            )

        index[unique_id] = asset

    return index


def make_asset_key4_for_special_lookup(asset):
    # For normal assets, use the first four parts of finalized unique_id.
    # For special assets, compute the same first-four identity without assigning
    # photo_library_asset_unique_id into the special asset object.
    unique_id = asset.get("photo_library_asset_unique_id")

    if unique_id is not None:
        return tuple(unique_id[:4])

    file_size = epl._get_file_size_from_asset(asset)
    adjustment_signature = epl._make_adjustment_signature(asset)

    unique_id = epl._make_photo_library_asset_unique_id(
        original_filename=asset.get("original_filename"),
        filename=asset.get("filename"),
        date=asset.get("date"),
        file_size=file_size,
        adjustment_signature=adjustment_signature,
        sha256=None,
    )

    if unique_id is None:
        return None

    return tuple(unique_id[:4])


def make_special_asset_key4_index(inventory, bucket_names=("SYNDICATED",)):
    index = {}

    special_assets = inventory.get("special_assets") or {}

    for bucket_name in bucket_names:
        for asset in special_assets.get(bucket_name, []):
            key4 = make_asset_key4_for_special_lookup(asset)

            if key4 is None:
                continue

            if key4 not in index:
                index[key4] = []

            index[key4].append(asset)

    return index


def make_album_title_index(inventory):
    index = {}

    for album in (inventory.get("albums") or {}).values():
        title = album.get("title")

        if title is None:
            continue

        if title not in index:
            index[title] = []

        index[title].append(album)

    return index


def make_folder_path_index(inventory):
    index = {}

    for folder in (inventory.get("folders") or {}).values():
        path = folder.get("path") or folder.get("title")

        if path is None:
            continue

        if path not in index:
            index[path] = []

        index[path].append(folder)

    return index


def compare_asset_existence(inventory_backup, inventory_current, diff_records):
    backup_assets = make_asset_index(inventory_backup)
    current_assets = make_asset_index(inventory_current)

    current_syndicated_by_key4 = make_special_asset_key4_index(
        inventory_current,
        bucket_names=("SYNDICATED",),
    )

    backup_ids = set(backup_assets)
    current_ids = set(current_assets)

    for asset_id in sort_asset_ids(backup_ids - current_ids):
        backup_asset = backup_assets[asset_id]
        asset_key4 = tuple(asset_id[:4])
        current_syndicated_matches = current_syndicated_by_key4.get(asset_key4, [])

        if current_syndicated_matches:
            add_diff(
                diff_records=diff_records,
                change_type=ChangeType.ASSET_PRESENT_IN_CURRENT_AS_SYNDICATED,
                scope="asset",
                backup_object=backup_asset,
                current_object=current_syndicated_matches,
                backup_value=asset_display_name(backup_asset),
                current_value=[
                    asset_display_name(current_asset)
                    for current_asset in current_syndicated_matches
                ],
                note=(
                    "Backup normal asset is not present in current normal assets, "
                    "but a matching current special_assets/SYNDICATED asset exists by "
                    "original_filename/date/file_size/adjustment_signature. "
                    "Do not treat as true current-missing."
                ),
            )
            continue

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ASSET_MISSING_FROM_CURRENT,
            scope="asset",
            backup_object=backup_asset,
            current_object=None,
            backup_value=asset_display_name(backup_asset),
            current_value=None,
            note=(
                "Asset exists in backup normal assets, and no matching current "
                "normal asset or current syndicated asset was found."
            ),
        )

    for asset_id in sort_asset_ids(current_ids - backup_ids):
        current_asset = current_assets[asset_id]

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ASSET_NEW_IN_CURRENT,
            scope="asset",
            backup_object=None,
            current_object=current_asset,
            backup_value=None,
            current_value=asset_display_name(current_asset),
            note="Asset exists in current normal assets but not in backup normal assets.",
        )


def compare_asset_metadata(inventory_backup, inventory_current, diff_records):
    backup_assets = make_asset_index(inventory_backup)
    current_assets = make_asset_index(inventory_current)

    common_ids = set(backup_assets) & set(current_assets)

    metadata_fields = [
        "description",
        "keywords",
        "favorite",
        "hidden",
    ]

    for asset_id in sort_asset_ids(common_ids):
        backup_asset = backup_assets[asset_id]
        current_asset = current_assets[asset_id]

        changed_fields = {}

        for field_name in metadata_fields:
            backup_value = normalize_scalar_value(backup_asset.get(field_name))
            current_value = normalize_scalar_value(current_asset.get(field_name))

            if backup_value == current_value:
                continue

            changed_fields[field_name] = {
                "backup": backup_value,
                "current": current_value,
            }

        if not changed_fields:
            continue

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ASSET_METADATA_CHANGED,
            scope="asset",
            backup_object=backup_asset,
            current_object=current_asset,
            backup_value=changed_fields,
            current_value=changed_fields,
            note="Same normal asset identity exists in both libraries, but metadata differs.",
        )


def compare_asset_album_membership(inventory_backup, inventory_current, diff_records):
    backup_assets = make_asset_index(inventory_backup)
    current_assets = make_asset_index(inventory_current)

    common_ids = set(backup_assets) & set(current_assets)

    for asset_id in sort_asset_ids(common_ids):
        backup_asset = backup_assets[asset_id]
        current_asset = current_assets[asset_id]

        backup_album_titles = set(asset_album_titles(backup_asset))
        current_album_titles = set(asset_album_titles(current_asset))

        if backup_album_titles == current_album_titles:
            continue

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ASSET_ALBUM_MEMBERSHIP_CHANGED,
            scope="asset",
            backup_object=backup_asset,
            current_object=current_asset,
            backup_value=sorted(backup_album_titles),
            current_value=sorted(current_album_titles),
            note="Same normal asset identity exists in both libraries, but album membership differs.",
        )


def compare_asset_folder_paths(inventory_backup, inventory_current, diff_records):
    backup_assets = make_asset_index(inventory_backup)
    current_assets = make_asset_index(inventory_current)

    common_ids = set(backup_assets) & set(current_assets)

    for asset_id in sort_asset_ids(common_ids):
        backup_asset = backup_assets[asset_id]
        current_asset = current_assets[asset_id]

        backup_paths = set(asset_folder_paths(backup_asset))
        current_paths = set(asset_folder_paths(current_asset))

        if backup_paths == current_paths:
            continue

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ASSET_FOLDER_PATHS_CHANGED,
            scope="asset",
            backup_object=backup_asset,
            current_object=current_asset,
            backup_value=sorted(backup_paths),
            current_value=sorted(current_paths),
            note="Same normal asset identity exists in both libraries, but folder paths differ.",
        )


def compare_album_existence_and_folder_paths(inventory_backup, inventory_current, diff_records):
    backup_album_index = make_album_title_index(inventory_backup)
    current_album_index = make_album_title_index(inventory_current)

    backup_titles = set(backup_album_index)
    current_titles = set(current_album_index)

    for title in sorted(backup_titles - current_titles):
        backup_albums = backup_album_index[title]

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ALBUM_MISSING_FROM_CURRENT,
            scope="album",
            backup_object=backup_albums,
            current_object=None,
            backup_value=[
                album_display_name(album)
                for album in backup_albums
            ],
            current_value=None,
            note="Album title exists in backup but not in current.",
        )

    for title in sorted(current_titles - backup_titles):
        current_albums = current_album_index[title]

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ALBUM_NEW_IN_CURRENT,
            scope="album",
            backup_object=None,
            current_object=current_albums,
            backup_value=None,
            current_value=[
                album_display_name(album)
                for album in current_albums
            ],
            note="Album title exists in current but not in backup.",
        )

    for title in sorted(backup_titles & current_titles):
        backup_albums = backup_album_index[title]
        current_albums = current_album_index[title]

        # If duplicate album titles exist, do not try to infer one-to-one folder
        # path changes here. Report existence only by title in this pass.
        if len(backup_albums) != 1 or len(current_albums) != 1:
            continue

        backup_album = backup_albums[0]
        current_album = current_albums[0]

        backup_paths = set(album_folder_paths(backup_album))
        current_paths = set(album_folder_paths(current_album))

        if backup_paths == current_paths:
            continue

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ALBUM_FOLDER_PATHS_CHANGED,
            scope="album",
            backup_object=backup_album,
            current_object=current_album,
            backup_value=sorted(backup_paths),
            current_value=sorted(current_paths),
            note="Album title exists in both libraries, but folder path differs.",
        )


def compare_folder_paths(inventory_backup, inventory_current, diff_records):
    backup_folder_index = make_folder_path_index(inventory_backup)
    current_folder_index = make_folder_path_index(inventory_current)

    backup_paths = set(backup_folder_index)
    current_paths = set(current_folder_index)

    for path in sorted(backup_paths - current_paths):
        backup_folders = backup_folder_index[path]

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.FOLDER_MISSING_FROM_CURRENT,
            scope="folder",
            backup_object=backup_folders,
            current_object=None,
            backup_value=[
                folder_display_name(folder)
                for folder in backup_folders
            ],
            current_value=None,
            note="Folder path exists in backup but not in current.",
        )

    for path in sorted(current_paths - backup_paths):
        current_folders = current_folder_index[path]

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.FOLDER_NEW_IN_CURRENT,
            scope="folder",
            backup_object=None,
            current_object=current_folders,
            backup_value=None,
            current_value=[
                folder_display_name(folder)
                for folder in current_folders
            ],
            note="Folder path exists in current but not in backup.",
        )


def compare_inventories(inventory_backup, inventory_current):
    diff_records = []

    compare_asset_existence(inventory_backup, inventory_current, diff_records)
    compare_asset_metadata(inventory_backup, inventory_current, diff_records)
    compare_asset_album_membership(inventory_backup, inventory_current, diff_records)
    compare_asset_folder_paths(inventory_backup, inventory_current, diff_records)
    compare_album_existence_and_folder_paths(inventory_backup, inventory_current, diff_records)
    compare_folder_paths(inventory_backup, inventory_current, diff_records)

    return diff_records


def summarize_diff_records(diff_records):
    print("diff record count:", len(diff_records))
    print()

    counts = Counter(
        record.get("change_type")
        for record in diff_records
    )

    print("change_type counts")
    print("-" * 80)

    for change_type, count in sorted(counts.items()):
        print(f"{change_type}: {count}")

    print()

    scope_counts = Counter(
        record.get("scope")
        for record in diff_records
    )

    print("scope counts")
    print("-" * 80)

    for scope, count in sorted(scope_counts.items()):
        print(f"{scope}: {count}")

In [16]:
# ============================================================
# Section 5: Run inventory comparison summary
# ============================================================

if not (backup_unique_ok and current_unique_ok):
    raise RuntimeError(
        "Photo Library asset unique ID audit failed. "
        "Do not run cross-library comparison yet."
    )

diff_records = compare_inventories(
    inventory_backup=inventory_backup,
    inventory_current=inventory_current,
)

summarize_diff_records(diff_records)

diff record count: 60527

change_type counts
--------------------------------------------------------------------------------
ALBUM_FOLDER_PATHS_CHANGED: 317
ALBUM_MISSING_FROM_CURRENT: 68
ALBUM_NEW_IN_CURRENT: 841
ASSET_ALBUM_MEMBERSHIP_CHANGED: 12622
ASSET_FOLDER_PATHS_CHANGED: 19169
ASSET_METADATA_CHANGED: 3959
ASSET_MISSING_FROM_CURRENT: 16
ASSET_NEW_IN_CURRENT: 23334
ASSET_PRESENT_IN_CURRENT_AS_SYNDICATED: 193
FOLDER_MISSING_FROM_CURRENT: 5
FOLDER_NEW_IN_CURRENT: 3

scope counts
--------------------------------------------------------------------------------
album: 1226
asset: 59293
folder: 8


In [19]:
# ============================================================
# Section 5A: Inspect ASSET_MISSING_FROM_CURRENT records
# ============================================================

from pathlib import Path
from collections import Counter

print("=" * 120)
print("Section 5A: Inspect ASSET_MISSING_FROM_CURRENT records")
print("=" * 120)


def get_unique_id_parts(asset):
    unique_id = asset.get("photo_library_asset_unique_id")

    if unique_id is None:
        return {
            "photo_library_asset_unique_id": None,
            "original_filename": None,
            "date": None,
            "file_size": None,
            "adjustment_signature": None,
            "sha256": None,
        }

    return {
        "photo_library_asset_unique_id": unique_id,
        "original_filename": unique_id[0] if len(unique_id) > 0 else None,
        "date": unique_id[1] if len(unique_id) > 1 else None,
        "file_size": unique_id[2] if len(unique_id) > 2 else None,
        "adjustment_signature": unique_id[3] if len(unique_id) > 3 else None,
        "sha256": unique_id[4] if len(unique_id) > 4 else None,
    }


def path_exists(path):
    if path is None:
        return False

    return Path(path).exists()


def file_size_on_disk(path):
    if path is None:
        return None

    path = Path(path)

    if not path.exists():
        return None

    return path.stat().st_size


def album_titles(asset):
    albums = asset.get("albums") or {}

    return sorted(
        album.get("title") or ""
        for album in albums.values()
        if album.get("title")
    )


def folder_paths(asset):
    folders = asset.get("folders") or {}

    return sorted(
        folder.get("path") or folder.get("title") or ""
        for folder in folders.values()
        if folder.get("path") or folder.get("title")
    )


def get_backup_asset_from_missing_record(record):
    asset = record.get("backup_object")

    if asset is None:
        print("Bad ASSET_MISSING_FROM_CURRENT record:")
        print(record)
        raise RuntimeError("ASSET_MISSING_FROM_CURRENT record has no backup_object.")

    return asset


def summarize_missing_asset_record(record, index):
    asset = get_backup_asset_from_missing_record(record)
    unique_id_parts = get_unique_id_parts(asset)
    path = asset.get("path")

    return {
        "index": index,
        "change_type": record.get("change_type"),
        "note": record.get("note"),

        "backup_asset": {
            "uuid": asset.get("uuid"),
            "photo_library_asset_unique_id": unique_id_parts["photo_library_asset_unique_id"],

            "unique_id_original_filename": unique_id_parts["original_filename"],
            "unique_id_date": unique_id_parts["date"],
            "unique_id_file_size": unique_id_parts["file_size"],
            "unique_id_adjustment_signature": unique_id_parts["adjustment_signature"],
            "unique_id_sha256": unique_id_parts["sha256"],

            "original_filename": asset.get("original_filename"),
            "filename": asset.get("filename"),
            "is_movie": asset.get("is_movie"),

            "date": asset.get("date"),
            "date_added": asset.get("date_added"),
            "date_modified": asset.get("date_modified"),

            "path_exists": path_exists(path),
            "path": path,
            "file_size_on_disk": file_size_on_disk(path),

            "width": asset.get("width"),
            "height": asset.get("height"),
            "original_width": asset.get("original_width"),
            "original_height": asset.get("original_height"),

            "hasadjustments": asset.get("hasadjustments"),
            "path_edited": asset.get("path_edited"),
            "path_live_photo": asset.get("path_live_photo"),
            "path_edited_live_photo": asset.get("path_edited_live_photo"),
            "is_live_photo": asset.get("is_live_photo"),

            "description": asset.get("description"),
            "keywords": list(asset.get("keywords") or []),
            "favorite": asset.get("favorite"),
            "hidden": asset.get("hidden"),

            "albums": album_titles(asset),
            "folders": folder_paths(asset),
        },
    }


missing_from_current_records = [
    record
    for record in diff_records
    if record.get("change_type") == "ASSET_MISSING_FROM_CURRENT"
]

missing_from_current_summaries = [
    summarize_missing_asset_record(record, index)
    for index, record in enumerate(missing_from_current_records, start=1)
]

print("matched record count:", len(missing_from_current_summaries))

quick_counts = {
    "is_movie": Counter(
        item["backup_asset"].get("is_movie")
        for item in missing_from_current_summaries
    ),
    "favorite": Counter(
        item["backup_asset"].get("favorite")
        for item in missing_from_current_summaries
    ),
    "hidden": Counter(
        item["backup_asset"].get("hidden")
        for item in missing_from_current_summaries
    ),
    "path_exists": Counter(
        item["backup_asset"].get("path_exists")
        for item in missing_from_current_summaries
    ),
    "hasadjustments": Counter(
        item["backup_asset"].get("hasadjustments")
        for item in missing_from_current_summaries
    ),
    "unique_id_sha256_present": Counter(
        item["backup_asset"].get("unique_id_sha256") is not None
        for item in missing_from_current_summaries
    ),
}

print()
print("Quick counts")
print("-" * 120)

for name, counter in quick_counts.items():
    print(f"{name}: {dict(counter)}")


for item in missing_from_current_summaries:
    asset = item["backup_asset"]

    print()
    print("=" * 120)
    print(f'{item["index"]:02d}. {asset.get("original_filename")}')
    print("=" * 120)

    print("change_type:", item["change_type"])
    print("note:", item["note"])

    print()
    print("BACKUP asset")
    print("-" * 120)

    print("uuid:", asset.get("uuid"))
    print("photo_library_asset_unique_id:", repr(asset.get("photo_library_asset_unique_id")))
    print("original_filename:", asset.get("original_filename"))
    print("filename:", asset.get("filename"))
    print("is_movie:", asset.get("is_movie"))

    print()
    print("unique_id parts:")
    print("  original_filename:", asset.get("unique_id_original_filename"))
    print("  date:", asset.get("unique_id_date"))
    print("  file_size:", asset.get("unique_id_file_size"))
    print("  adjustment_signature:", asset.get("unique_id_adjustment_signature"))
    print("  sha256:", asset.get("unique_id_sha256"))

    print()
    print("date:", asset.get("date"))
    print("date_added:", asset.get("date_added"))
    print("date_modified:", asset.get("date_modified"))

    print()
    print("path_exists:", asset.get("path_exists"))
    print("path:", asset.get("path"))
    print("file_size_on_disk:", asset.get("file_size_on_disk"))

    print()
    print("width x height:", asset.get("width"), "x", asset.get("height"))
    print(
        "original_width x original_height:",
        asset.get("original_width"),
        "x",
        asset.get("original_height"),
    )

    print()
    print("hasadjustments:", asset.get("hasadjustments"))
    print("path_edited:", asset.get("path_edited"))
    print("path_live_photo:", asset.get("path_live_photo"))
    print("path_edited_live_photo:", asset.get("path_edited_live_photo"))
    print("is_live_photo:", asset.get("is_live_photo"))

    print()
    print("description:", asset.get("description"))
    print("keywords:", asset.get("keywords"))
    print("favorite:", asset.get("favorite"))
    print("hidden:", asset.get("hidden"))

    print()
    print("albums:")
    if asset.get("albums"):
        for album in asset["albums"]:
            print("  -", album)
    else:
        print("  []")

    print("folders:")
    if asset.get("folders"):
        for folder in asset["folders"]:
            print("  -", folder)
    else:
        print("  []")

Section 5A: Inspect ASSET_MISSING_FROM_CURRENT records
matched record count: 16

Quick counts
------------------------------------------------------------------------------------------------------------------------
is_movie: {False: 16}
favorite: {False: 16}
hidden: {False: 16}
path_exists: {True: 16}
hasadjustments: {True: 5, False: 11}
unique_id_sha256_present: {False: 16}

01. IMG_0078.PNG
change_type: ASSET_MISSING_FROM_CURRENT
note: Asset exists in backup normal assets, and no matching current normal asset or current syndicated asset was found.

BACKUP asset
------------------------------------------------------------------------------------------------------------------------
uuid: E5FDDA31-773F-4327-8480-C554EC99DCEB
photo_library_asset_unique_id: ('IMG_0078.PNG', '12-15 14:01:41.00', 6855252, (1284, 2100), None)
original_filename: IMG_0078.PNG
filename: E5FDDA31-773F-4327-8480-C554EC99DCEB.png
is_movie: False

unique_id parts:
  original_filename: IMG_0078.PNG
  date: 12-15 14:

In [ ]:
# ============================================================
# Section 5A: Compact ASSET_MISSING_FROM_CURRENT review list
# ============================================================

from collections import defaultdict, Counter
from pathlib import Path
import explorephotoslibrary as epl


TARGET_CHANGE_TYPE = "ASSET_MISSING_FROM_CURRENT"


def format_bytes(value):
    if value is None:
        return "None"

    return f"{value:,} bytes"


def path_exists(path):
    if path is None:
        return False

    return Path(path).exists()


def asset_unique_id_parts(asset):
    unique_id = asset.get("photo_library_asset_unique_id")

    if unique_id is None:
        return (None, None, None, None, None)

    return tuple(unique_id)


def asset_key4(asset):
    unique_id = asset.get("photo_library_asset_unique_id")

    if unique_id is not None:
        return tuple(unique_id[:4])

    file_size = epl._get_file_size_from_asset(asset)
    adjustment_signature = epl._make_adjustment_signature(asset)

    unique_id = epl._make_photo_library_asset_unique_id(
        original_filename=asset.get("original_filename"),
        filename=asset.get("filename"),
        date=asset.get("date"),
        file_size=file_size,
        adjustment_signature=adjustment_signature,
        sha256=None,
    )

    if unique_id is None:
        return None

    return tuple(unique_id[:4])


def asset_album_titles_for_print(asset):
    albums = asset.get("albums") or {}

    titles = [
        album.get("title")
        for album in albums.values()
        if album.get("title")
    ]

    return titles


def asset_folder_paths_for_print(asset):
    folders = asset.get("folders") or {}

    paths = [
        folder.get("path") or folder.get("title")
        for folder in folders.values()
        if folder.get("path") or folder.get("title")
    ]

    return paths


def short_text(value, limit=90):
    if value is None:
        return "-"

    text = str(value).replace("\n", " ")

    if len(text) <= limit:
        return text

    return text[:limit] + "..."


def first_or_dash(values, limit=90):
    if not values:
        return "-"

    return short_text(values[0], limit=limit)


def build_current_original_filename_index(inventory_current):
    index = defaultdict(list)

    for asset in inventory_current.get("assets") or []:
        original_filename = asset.get("original_filename")

        if original_filename:
            index[original_filename].append(("NORMAL", asset))

    for bucket_name, special_assets in (inventory_current.get("special_assets") or {}).items():
        for asset in special_assets:
            original_filename = asset.get("original_filename")

            if original_filename:
                index[original_filename].append((bucket_name, asset))

    return index


def current_candidate_brief(bucket_name, asset):
    key4 = asset_key4(asset)

    if key4 is None:
        date_key = None
        file_size = None
        adjustment_signature = None
    else:
        _, date_key, file_size, adjustment_signature = key4

    return (
        f"{bucket_name}"
        f" | uuid={asset.get('uuid')}"
        f" | date_key={date_key}"
        f" | file_size={format_bytes(file_size)}"
        f" | adjustment_signature={adjustment_signature}"
        f" | size={asset.get('width')}x{asset.get('height')}"
        f" | hasadjustments={asset.get('hasadjustments')}"
        f" | path_exists={path_exists(asset.get('path'))}"
    )


missing_records = [
    record
    for record in diff_records
    if record.get("change_type") == TARGET_CHANGE_TYPE
]

current_by_original_filename = build_current_original_filename_index(inventory_current)

print("=" * 120)
print("Section 5A: Compact ASSET_MISSING_FROM_CURRENT review list")
print("=" * 120)
print("matched record count:", len(missing_records))
print()

quick_counts = {
    "is_movie": Counter(bool(record.get("backup_object", {}).get("is_movie")) for record in missing_records),
    "hasadjustments": Counter(bool(record.get("backup_object", {}).get("hasadjustments")) for record in missing_records),
    "favorite": Counter(bool(record.get("backup_object", {}).get("favorite")) for record in missing_records),
    "hidden": Counter(bool(record.get("backup_object", {}).get("hidden")) for record in missing_records),
    "path_exists": Counter(path_exists(record.get("backup_object", {}).get("path")) for record in missing_records),
}

print("Quick counts")
print("-" * 120)

for name, counter in quick_counts.items():
    print(name + ":", dict(counter))

print()
print("=" * 120)
print("One-by-one manual verification checklist")
print("=" * 120)

for index, record in enumerate(missing_records, start=1):
    backup_asset = record.get("backup_object") or {}
    unique_id = backup_asset.get("photo_library_asset_unique_id")
    key4 = asset_key4(backup_asset)

    original_filename = backup_asset.get("original_filename")
    current_candidates = current_by_original_filename.get(original_filename, [])

    if key4 is None:
        date_key = None
        file_size = None
        adjustment_signature = None
    else:
        _, date_key, file_size, adjustment_signature = key4

    album_titles = asset_album_titles_for_print(backup_asset)
    folder_paths = asset_folder_paths_for_print(backup_asset)

    print()
    print(f"{index:02d}. {original_filename}")
    print("-" * 120)
    print("change_type:", record.get("change_type"))
    print("description:", record.get("description"))
    print("note:", record.get("note"))
    print("uuid:", backup_asset.get("uuid"))
    print("unique_id:", repr(unique_id))
    print("date:", backup_asset.get("date"))
    print("date_added:", backup_asset.get("date_added"))
    print("date_modified:", backup_asset.get("date_modified"))
    print("file_size:", format_bytes(file_size))
    print("size:", f"{backup_asset.get('width')}x{backup_asset.get('height')}")
    print("original_size:", f"{backup_asset.get('original_width')}x{backup_asset.get('original_height')}")
    print("hasadjustments:", backup_asset.get("hasadjustments"))
    print("adjustment_signature:", adjustment_signature)
    print("path_exists:", path_exists(backup_asset.get("path")))
    print("path:", backup_asset.get("path"))
    print("path_edited:", backup_asset.get("path_edited"))
    print("description_text:", short_text(backup_asset.get("description"), limit=140))
    print("keywords:", list(backup_asset.get("keywords") or []))
    print("first_album:", first_or_dash(album_titles, limit=140))
    print("all_albums:", album_titles)
    print("folders:", folder_paths)

    print("current candidates with same original_filename:", len(current_candidates))

    for candidate_index, (bucket_name, current_asset) in enumerate(current_candidates[:5], start=1):
        print(
            f"  current_candidate_{candidate_index}:",
            current_candidate_brief(bucket_name, current_asset),
        )

    if len(current_candidates) > 5:
        print("  ... more current candidates not printed")

target change type: ASSET_MISSING_FROM_CURRENT
matched record count: 16

change_type: ASSET_MISSING_FROM_CURRENT


KeyError: 'change_type_description'

In [14]:
# ============================================================
# Appendix A: Duplicate diagnostic archive
# 
# TEMP: Diagnose potential duplicate groups by SHA256
#       with full manual-review metadata
# ============================================================

import hashlib
import os
import time
from datetime import datetime


def compute_sha256_for_asset(asset, chunk_size=1024 * 1024):
    cached_sha256 = asset.get("content_sha256")
    if cached_sha256:
        return cached_sha256

    path = asset.get("path")

    if path is None:
        return None

    if not os.path.exists(path):
        return None

    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    digest = sha256.hexdigest()
    asset["content_sha256"] = digest
    return digest


def build_potential_duplicate_groups_by_unique_id(inventory):
    unique_id_to_assets = {}

    for asset in inventory["assets"]:
        unique_id = asset.get("photo_library_asset_unique_id")

        if unique_id is None:
            continue

        if unique_id not in unique_id_to_assets:
            unique_id_to_assets[unique_id] = []

        unique_id_to_assets[unique_id].append(asset)

    return {
        unique_id: assets
        for unique_id, assets in unique_id_to_assets.items()
        if len(assets) > 1
    }


def normalize_string_list(values):
    result = []

    if values is None:
        return result

    if isinstance(values, str):
        return [values]

    if isinstance(values, dict):
        iterable = values.values()
    elif isinstance(values, (list, tuple, set)):
        iterable = values
    else:
        return [str(values)]

    for item in iterable:
        if item is None:
            continue

        if isinstance(item, str):
            value = item
        elif isinstance(item, dict):
            value = (
                item.get("title")
                or item.get("name")
                or item.get("path")
                or item.get("folder_path")
                or item.get("album_path")
            )
        else:
            value = str(item)

        if value:
            result.append(value)

    return sorted(set(result))


def get_asset_album_titles(asset):
    albums = asset.get("albums")
    return normalize_string_list(albums)


def get_asset_folder_paths(asset):
    folders = asset.get("folders")
    folder_paths = normalize_string_list(folders)

    # Some inventory formats may store folder paths under different keys.
    extra_candidates = [
        asset.get("folder_paths"),
        asset.get("folder_path"),
        asset.get("album_folder_paths"),
    ]

    for candidate in extra_candidates:
        folder_paths.extend(normalize_string_list(candidate))

    return sorted(set(folder_paths))


def get_asset_keywords(asset):
    keywords = asset.get("keywords")
    return normalize_string_list(keywords)


def get_asset_description(asset):
    return (
        asset.get("description")
        or asset.get("caption")
        or asset.get("title")
        or ""
    )


def parse_date_added_for_sort(asset):
    date_added = asset.get("date_added")

    if not date_added:
        return datetime.max

    if isinstance(date_added, datetime):
        return date_added

    text = str(date_added)

    try:
        return datetime.fromisoformat(text.replace("Z", "+00:00"))
    except Exception:
        return datetime.max


def asset_metadata_signature(asset):
    return {
        "albums": tuple(get_asset_album_titles(asset)),
        "folders": tuple(get_asset_folder_paths(asset)),
        "keywords": tuple(get_asset_keywords(asset)),
        "description": get_asset_description(asset),
        "favorite": asset.get("favorite"),
        "hidden": asset.get("hidden"),
        "hasadjustments": asset.get("hasadjustments"),
        "adjustment_signature": asset.get("adjustment_signature"),
    }


def metadata_score(asset):
    return (
        len(get_asset_album_titles(asset)) * 10
        + len(get_asset_folder_paths(asset)) * 10
        + len(get_asset_keywords(asset)) * 5
        + (1 if get_asset_description(asset) else 0)
        + (1 if asset.get("favorite") else 0)
        + (1 if asset.get("hidden") else 0)
    )


def choose_representative_asset(assets):
    # Prefer metadata-rich assets; tie-break by earliest Date Added.
    return sorted(
        assets,
        key=lambda asset: (
            -metadata_score(asset),
            parse_date_added_for_sort(asset),
            asset.get("uuid") or "",
        ),
    )[0]


def print_asset_manual_review_block(asset, indent="  "):
    print(f"{indent}UUID:", asset.get("uuid"))
    print(f"{indent}Original File Name:", asset.get("original_filename"))
    print(f"{indent}Filename:", asset.get("filename"))
    print(f"{indent}Date:", asset.get("date"))
    print(f"{indent}Date Added:", asset.get("date_added"))
    print(f"{indent}File Size:", asset.get("file_size_bytes"))
    print(f"{indent}Has Adjustments:", asset.get("hasadjustments"))
    print(f"{indent}Adjustment Signature:", asset.get("adjustment_signature"))
    print(f"{indent}Width x Height:", asset.get("width"), "x", asset.get("height"))
    print(f"{indent}Original Width x Height:", asset.get("original_width"), "x", asset.get("original_height"))
    print(f"{indent}Albums:", get_asset_album_titles(asset))
    print(f"{indent}Folder Paths:", get_asset_folder_paths(asset))
    print(f"{indent}Keywords:", get_asset_keywords(asset))
    print(f"{indent}Description:", get_asset_description(asset))
    print(f"{indent}Favorite:", asset.get("favorite"))
    print(f"{indent}Hidden:", asset.get("hidden"))
    print(f"{indent}Path:", asset.get("path"))


def print_cleanup_recommendation(assets):
    metadata_signatures = [asset_metadata_signature(asset) for asset in assets]
    metadata_all_same = all(
        signature == metadata_signatures[0]
        for signature in metadata_signatures
    )

    representative = choose_representative_asset(assets)

    if metadata_all_same:
        print("Recommendation:")
        print("  Metadata appears identical.")
        print("  Keep earliest / representative asset:")
        print("   ", representative.get("uuid"))
        print("  Delete other duplicate asset(s):")
        for asset in assets:
            if asset is not representative:
                print("   ", asset.get("uuid"))
    else:
        print("Recommendation:")
        print("  Metadata differs across duplicate assets.")
        print("  Do NOT blindly delete.")
        print("  Suggested representative, based on richer metadata + earliest Date Added:")
        print("   ", representative.get("uuid"))
        print("  Before deleting others, manually confirm whether album/folder/keyword membership should be preserved.")


def diagnose_potential_duplicate_groups_with_sha256_and_metadata(
    inventory,
    label,
    max_true_duplicate_groups_to_print=50,
    max_key_collision_groups_to_print=20,
):
    start_time = time.perf_counter()

    potential_groups = build_potential_duplicate_groups_by_unique_id(inventory)

    true_content_duplicate_groups = []
    key_collision_groups = []
    sha_error_assets = []

    checked_asset_count = 0

    for unique_id, assets in potential_groups.items():
        sha256_to_assets = {}

        for asset in assets:
            checked_asset_count += 1
            sha256 = compute_sha256_for_asset(asset)

            if sha256 is None:
                sha_error_assets.append(asset)
                continue

            if sha256 not in sha256_to_assets:
                sha256_to_assets[sha256] = []

            sha256_to_assets[sha256].append(asset)

        duplicate_sha_groups = {
            sha256: sha_assets
            for sha256, sha_assets in sha256_to_assets.items()
            if len(sha_assets) > 1
        }

        if duplicate_sha_groups:
            for sha256, sha_assets in duplicate_sha_groups.items():
                true_content_duplicate_groups.append(
                    {
                        "unique_id": unique_id,
                        "sha256": sha256,
                        "assets": sha_assets,
                    }
                )

        if len(sha256_to_assets) > 1:
            key_collision_groups.append(
                {
                    "unique_id": unique_id,
                    "sha256_to_assets": sha256_to_assets,
                }
            )

    elapsed = time.perf_counter() - start_time

    print(label)
    print("-" * 120)
    print("potential duplicate unique_id group count:", len(potential_groups))
    print("checked asset count:", checked_asset_count)
    print("sha error asset count:", len(sha_error_assets))
    print("true content duplicate group count:", len(true_content_duplicate_groups))
    print("key collision group count:", len(key_collision_groups))
    print("elapsed seconds:", round(elapsed, 3))

    print()
    print("TRUE CONTENT DUPLICATE GROUPS — MANUAL REVIEW")
    print("-" * 120)

    for index, group in enumerate(true_content_duplicate_groups, start=1):
        if index > max_true_duplicate_groups_to_print:
            print("... more true content duplicate groups not printed")
            break

        assets_sorted = sorted(
            group["assets"],
            key=lambda asset: (
                parse_date_added_for_sort(asset),
                asset.get("uuid") or "",
            ),
        )

        print("=" * 120)
        print(f"Group {index:02d}")
        print("=" * 120)
        print("unique_id:", group["unique_id"])
        print("sha256:", group["sha256"])
        print("asset count:", len(assets_sorted))

        first_asset = assets_sorted[0]
        print("Original File Name:", first_asset.get("original_filename"))
        print("Date:", first_asset.get("date"))
        print("File Size:", first_asset.get("file_size_bytes"))
        print("Adjustment Signature:", first_asset.get("adjustment_signature"))

        union_albums = sorted(
            set(
                album
                for asset in assets_sorted
                for album in get_asset_album_titles(asset)
            )
        )
        union_folders = sorted(
            set(
                folder
                for asset in assets_sorted
                for folder in get_asset_folder_paths(asset)
            )
        )
        union_keywords = sorted(
            set(
                keyword
                for asset in assets_sorted
                for keyword in get_asset_keywords(asset)
            )
        )

        print("Union Albums:", union_albums)
        print("Union Folder Paths:", union_folders)
        print("Union Keywords:", union_keywords)

        print()
        print_cleanup_recommendation(assets_sorted)
        print()

        for asset_index, asset in enumerate(assets_sorted, start=1):
            print("-" * 120)
            print(f"Asset {asset_index}")
            print_asset_manual_review_block(asset, indent="  ")

        print()

    print()
    print("KEY COLLISION GROUPS")
    print("-" * 120)

    for index, group in enumerate(key_collision_groups, start=1):
        if index > max_key_collision_groups_to_print:
            print("... more key collision groups not printed")
            break

        print("=" * 120)
        print(f"Key Collision Group {index:02d}")
        print("=" * 120)
        print("unique_id:", group["unique_id"])
        print("sha256 count:", len(group["sha256_to_assets"]))

        for sha256, assets in group["sha256_to_assets"].items():
            print("  sha256:", sha256)
            print("  asset count:", len(assets))

            for asset in assets:
                print("    uuid:", asset.get("uuid"))
                print("    original_filename:", asset.get("original_filename"))
                print("    filename:", asset.get("filename"))
                print("    date:", asset.get("date"))
                print("    date_added:", asset.get("date_added"))
                print("    file_size_bytes:", asset.get("file_size_bytes"))
                print("    albums:", get_asset_album_titles(asset))
                print("    folder_paths:", get_asset_folder_paths(asset))
                print("    keywords:", get_asset_keywords(asset))
                print("    path:", asset.get("path"))

        print()

    return {
        "potential_groups": potential_groups,
        "true_content_duplicate_groups": true_content_duplicate_groups,
        "key_collision_groups": key_collision_groups,
        "sha_error_assets": sha_error_assets,
    }


backup_duplicate_diagnostic = diagnose_potential_duplicate_groups_with_sha256_and_metadata(
    inventory_backup,
    "BACKUP potential duplicate diagnostic with metadata",
)

print()

current_duplicate_diagnostic = diagnose_potential_duplicate_groups_with_sha256_and_metadata(
    inventory_current,
    "CURRENT potential duplicate diagnostic with metadata",
)

BACKUP potential duplicate diagnostic with metadata
------------------------------------------------------------------------------------------------------------------------
potential duplicate unique_id group count: 0
checked asset count: 0
sha error asset count: 0
true content duplicate group count: 0
key collision group count: 0
elapsed seconds: 0.053

TRUE CONTENT DUPLICATE GROUPS — MANUAL REVIEW
------------------------------------------------------------------------------------------------------------------------

KEY COLLISION GROUPS
------------------------------------------------------------------------------------------------------------------------

CURRENT potential duplicate diagnostic with metadata
------------------------------------------------------------------------------------------------------------------------
potential duplicate unique_id group count: 0
checked asset count: 0
sha error asset count: 0
true content duplicate group count: 0
key collision group count: 

In [9]:
# ============================================================
# Appendix B: Debug assets without unique ID
# 
# DEBUG: Dump assets without photo_library_asset_unique_id
# ============================================================

import os
import time
from collections import Counter

def debug_dump_assets_without_photo_library_asset_unique_id(inventory, label, max_print=80):
    missing_assets = [
        asset
        for asset in inventory["assets"]
        if asset.get("photo_library_asset_unique_id") is None
    ]

    reason_counter = Counter()

    print(label)
    print("-" * 120)
    print("assets without photo_library_asset_unique_id:", len(missing_assets))
    print()

    for asset in missing_assets:
        path = asset.get("path")
        original_filename = asset.get("original_filename")
        filename = asset.get("filename")
        date = asset.get("date")
        file_size_bytes = asset.get("file_size_bytes")
        adjustment_signature = asset.get("adjustment_signature")

        if original_filename is None and filename is None:
            reason_counter["missing filename and original_filename"] += 1

        if date is None:
            reason_counter["missing date"] += 1

        if path is None:
            reason_counter["path is None"] += 1
        elif not os.path.exists(path):
            reason_counter["path does not exist"] += 1

        if file_size_bytes is None:
            reason_counter["file_size_bytes is None"] += 1

        if adjustment_signature is None:
            reason_counter["adjustment_signature is None"] += 1

    print("reason counter:")
    for reason, count in reason_counter.most_common():
        print(f"  {reason}: {count}")

    print()
    print("missing asset details:")
    print("-" * 120)

    for index, asset in enumerate(missing_assets[:max_print], start=1):
        path = asset.get("path")

        print(f"{index:02d}.")
        print("  uuid:", asset.get("uuid"))
        print("  original_filename:", asset.get("original_filename"))
        print("  filename:", asset.get("filename"))
        print("  date:", asset.get("date"))
        print("  date_added:", asset.get("date_added"))
        print("  path:", path)
        print("  path_exists:", None if path is None else os.path.exists(path))
        print("  file_size_bytes:", asset.get("file_size_bytes"))
        print("  adjustment_signature:", asset.get("adjustment_signature"))
        print("  is_movie:", asset.get("is_movie"))
        print("  hasadjustments:", asset.get("hasadjustments"))
        print("  path_edited:", asset.get("path_edited"))
        print("  asset_scope:", asset.get("asset_scope"))
        print("  albums:", list((asset.get("albums") or {}).values()))
        print("  folders:", list((asset.get("folders") or {}).values()))
        print("-" * 120)

debug_dump_assets_without_photo_library_asset_unique_id(
    inventory_current,
    "CURRENT DEFAULT assets without photo_library_asset_unique_id",
)

CURRENT DEFAULT assets without photo_library_asset_unique_id
------------------------------------------------------------------------------------------------------------------------
assets without photo_library_asset_unique_id: 0

reason counter:

missing asset details:
------------------------------------------------------------------------------------------------------------------------
